In [1]:
:! attribute t =
    `(
        backend_switch `(()) `({}) {
            Gleam = (\ () => global "import lustre/attribute") : () -> ()
        }
        $'' : $'attribute.Attribute(`t)'
    )

:! element t =
    `(
        backend_switch `(()) `({}) {
            Gleam = (\ () => global "import lustre/element") : () -> ()
        }
        $'' : $'element.Element(`t)'
    )

:= text \* t. (text : string) : element t =
    global "import lustre/element/html"
    $'element.text(!text)'

:= on_click \* t. (msg : t) : attribute t =
    global "import lustre/event"
    $'event.on_click(!msg)'

:= style \* t. (styles : list (string * string)) : attribute t =
    global "import lustre/attribute"
    := styles = styles |> listm'::box
    $'attribute.styles(!styles)'

:= class \* t. (name : string) : attribute t =
    global "import lustre/attribute"
    $'attribute.class(!name)'

:= role \* t. (name : string) : attribute t =
    global "import lustre/attribute"
    $'attribute.role(!name)'

:= attr \* t. (name : string) (value : string) : attribute t =
    global "import lustre/attribute"
    $'attribute.attribute(!name, !value)'

:= div \* t. (attributes : list (attribute t)) (children : list (element t)) : element t =
    global "import lustre/element/html"
    := attributes = attributes |> listm'::box
    := children = children |> listm'::box
    $'html.div(!attributes, !children)'

:= p \* t. (attributes : list (attribute t)) (children : list (element t)) : element t =
    global "import lustre/element/html"
    := attributes = attributes |> listm'::box
    := children = children |> listm'::box
    $'html.p(!attributes, !children)'

:= button \* t. (attributes : list (attribute t)) (children : list (element t)) : element t =
    global "import lustre/element/html"
    := attributes = (class "lustre-ui-button" :: role "button" :: attr "tabindex" "0" :: attributes) |> listm'::box
    := children = children |> listm'::box
    $'html.button(!attributes, !children)'

:= centre \* t. (attributes : list (attribute t)) (children : element t) : element t =
    div (class "lustre-ui-centre" :: attributes) [ children ]

:! app model msg = $'lustre.App(`model, `model, `msg)'

:= simple \* model msg.
    (init : model -> model)
    (update : model * msg -> model)
    (view : model -> element msg)
    : app model msg
    =
    global "import lustre"
    $'lustre.simple(!init, fn (a, b) { !update(##(a, b)) }, !view)'

:= to_html \* t. (element : element t) : string =
    $'element.to_string(!element)'

:= start id app =
    $'let assert Ok(_) = lustre.start(!app, !id, 0)' : ()

:== model = int

:= init initial_count : model =
    ? initial_count < 0
    ?> 0
    |: initial_count

:| msg =
    | Incr
    | Decr

:= update n (model, msg) =
    ?= msg ?|
    | Incr => model + n
    | Decr => model - n

:= view model =
    div [
        style [
            "display", "flex"
        ]
    ] [
        div [
            style [
                "display", "flex"
                "flex-direction", "column"
            ]
        ] [
            button [
                on_click Incr
            ] [
                text "+"
            ]
            p [
                style [
                    "text-align", "center"
                ]
            ] [
                model |> sm'::obj_to_string |> text
            ]
            button [
                on_click Decr
            ] [
                text "-"
            ]
        ]
    ]

:< sm'_operators

:= expect_html (expected : string) (actual : string) : () =
    ? actual <> expected
    ?> failwith ("__assert_eq { actual = " ++# actual ++# "; expected = " ++# expected ++# " }")

#### tests (test)

In [2]:
///- --test --symbol
///> gleam -t javascript -d lustre=\">=5.7.0\" gleam_json=\">=2.0.0\"

:= model = init (dyn -3i32)
:= model = update 1i32 (model, Incr)
:= model = update 1i32 (model, Incr)
:= model = update 1i32 (model, Decr)
model
|> view
|> to_html
|> expect_html (
    "<div style=\"display:flex;\"><div style=\"display:flex;flex-direction:column;\"><button "
    ^^ "class=\"lustre-ui-button\" role=\"button\" tabindex=\"0\">+</button><p "
    ^^ "style=\"text-align:center;\">1</p><button class=\"lustre-ui-button\" role=\"button\" "
    ^^ "tabindex=\"0\">-</button></div></div>"
)

In [3]:
///- --test --symbol
///> gleam -t javascript -d lustre=\">=5.7.0\" gleam_json=\">=2.0.0\"

:= model = init (dyn 5i32)
:= model = update 2i32 (model, Decr)
model
|> view
|> to_html
|> expect_html (
    "<div style=\"display:flex;\"><div style=\"display:flex;flex-direction:column;\"><button "
    ^^ "class=\"lustre-ui-button\" role=\"button\" tabindex=\"0\">+</button><p "
    ^^ "style=\"text-align:center;\">3</p><button class=\"lustre-ui-button\" role=\"button\" "
    ^^ "tabindex=\"0\">-</button></div></div>"
)